# Aspect-Level Sentiment Analysis (Lexicon-Based) - Ulasan Aplikasi
Pengukuran skor sentimen tingkat aspek secara lokal pada tingkat klausa menggunakan kamus sentimen Bahasa Indonesia (InSet Lexicon).

In [ ]:
import pandas as pd
import numpy as np
import re
import os

## 1. Load Data & Lexicon

In [ ]:
dataset_path = '../data/cleaned_app_reviews_with_aspect_labels.csv'
df = pd.read_csv(dataset_path).dropna(subset=['text_clean', 'review_text'])

# Muat kamus sentimen
pos_path = os.path.join('..', '..', '..', 'modeling', 'data', 'positive.tsv')
neg_path = os.path.join('..', '..', '..', 'modeling', 'data', 'negative.tsv')

if os.path.exists(pos_path) and os.path.exists(neg_path):
    pos_df = pd.read_csv(pos_path, sep='\t')
    neg_df = pd.read_csv(neg_path, sep='\t')
    
    lexicon = {}
    for _, row in pos_df.iterrows():
        lexicon[str(row['word']).lower()] = float(row['weight'])
    for _, row in neg_df.iterrows():
        lexicon[str(row['word']).lower()] = float(row['weight'])
    print(f"Lexicon loaded. Total {len(lexicon)} kata.")
else:
    print("File lexicon tidak ditemukan!")

## 2. Inisialisasi Keyword Aspek

In [ ]:
aspect_keywords = {
    'aspek_rasa': ['kopi', 'minuman', 'rasa', 'enak', 'pahit', 'manis', 'es', 'susu', 'roti', 'menu', 'cup', 'kualitas', 'asin', 'gurih', 'cokelat', 'latte', 'matcha', 'boba', 'tawar', 'encer', 'pekat', 'panas', 'dingin'],
    'aspek_harga': ['harga', 'mahal', 'murah', 'promo', 'diskon', 'cashback', 'ovo', 'gopay', 'shopeepay', 'worth', 'hemat', 'paket', 'voucher', 'poin', 'reward', 'points', 'point', 'vocher', 'vocr', 'dana', 'qris', 'linkaja'],
    'aspek_aplikasi': ['app', 'aplikasi', 'order', 'pesan', 'sistem', 'down', 'error', 'apk', 'bug', 'crash', 'lag', 'login', 'logout', 'update', 'notif', 'ui', 'tampilan', 'lemot', 'macet', 'otp', 'sms', 'keluar', 'masuk', 'loading', 'pemuatan', 'uninstall', 'uninstal', 'copot'],
    'aspek_pelayanan': ['cs', 'customer', 'service', 'komplain', 'respon', 'balas', 'admin', 'pelayanan', 'staff', 'karyawan', 'ramah', 'sopan', 'jutek', 'barista', 'kasir'],
    'aspek_kecepatan': ['lama', 'cepat', 'antri', 'nunggu', 'lambat', 'tunggu', 'antrean', 'gercep', 'lelet', 'durasi', 'menit', 'jam', 'telat', 'keterlambatan'],
    'aspek_stok': ['habis', 'stok', 'menu', 'sedia', 'kosong', 'varian', 'kehabisan', 'ready', 'sold', 'out'],
    'aspek_pengiriman': ['kurir', 'driver', 'gojek', 'grab', 'lalamove', 'kirim', 'tumpah', 'packaging', 'kemasan', 'bocor', 'solasi', 'segel'],
    'aspek_pesanan': ['salah', 'kurang', 'tidak sesuai', 'beda', 'cancel', 'refund', 'batal', 'kembali uang', 'salah pesanan']
}

## 3. Ekstraksi Klausa & Scoring Sentimen Aspek
Ulasan dipecah menjadi klausa berdasarkan tanda baca dan kata hubung. Skor sentimen aspek dihitung pada klausa yang memuat kata kunci aspek tersebut.

In [ ]:
def split_into_clauses(text):
    if not isinstance(text, str):
        return []
    parts = re.split(r'[.,;!?\n]+', text)
    clauses = []
    for p in parts:
        subparts = re.split(r'\b(tapi|namun|tetapi|sedangkan|padahal|meskipun|walaupun|lalu|kemudian)\b', p, flags=re.IGNORECASE)
        clauses.extend([sp.strip() for sp in subparts if sp.strip()])
    return clauses

def calculate_lexicon_score(text, lexicon_dict):
    words = str(text).lower().split()
    score = 0.0
    for word in words:
        if word in lexicon_dict:
            score += lexicon_dict[word]
    return score

def get_aspect_sentiment(row, aspect_name, keywords, lexicon_dict):
    if row[aspect_name] == 0:
        return "None"
    
    original_text = row['review_text']
    clauses = split_into_clauses(original_text)
    kws = keywords[aspect_name]
    
    # Filter klausa yang cocok dengan kata kunci aspek
    matched_clauses = [c for c in clauses if any(kw in c.lower() for kw in kws)]
    
    # Gunakan text_clean jika tidak ada klausa spesifik yang terdeteksi
    target_text = " ".join(matched_clauses) if matched_clauses else row['text_clean']
    score = calculate_lexicon_score(target_text, lexicon_dict)
    
    if score > 0:
        return "positive"
    elif score < 0:
        return "negative"
    else:
        return "neutral"

## 4. Eksekusi Sentimen Aspek

In [ ]:
print("Menghitung sentimen aspek...")
for aspect in aspect_keywords.keys():
    sentiment_col = aspect.replace('aspek_', 'sentimen_')
    df[sentiment_col] = df.apply(
        lambda row: get_aspect_sentiment(row, aspect, aspect_keywords, lexicon), 
        axis=1
    )
print("Selesai.")

## 5. Analisis Hasil Sentimen

In [ ]:
for aspect in aspect_keywords.keys():
    sentiment_col = aspect.replace('aspek_', 'sentimen_')
    counts = df[df[sentiment_col] != 'None'][sentiment_col].value_counts()
    print(f"\nDistribusi Sentimen - Aspek {aspect.replace('aspek_', '').upper()}:")
    print(counts)

## 6. Simpan Dataset Sentimen Aspek

In [ ]:
os.makedirs('../data', exist_ok=True)
output_path = '../data/reviews_with_aspect_sentiment.csv'
df.to_csv(output_path, index=False)
print(f"Dataset disimpan di: {output_path}")